In [1]:
%pip install --upgrade openai python-dotenv


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
def load_prompt(role):
    with open(f"{role}_engineer.txt", "r") as file:
        return file.read()

In [4]:
def call_openai(user_prompt, role_file_name):
    system_prompt = load_prompt(role_file_name)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0,
    )
    return response.choices[0].message.content

In [5]:
def requirements_engineer(request):
    prompt = f"Customer has requested: {request}. As a requirements engineer, list all use-cases and detailed requirements. Include productivity estimation."
    return call_openai(prompt, "requirements")

def system_engineer(request):
    prompt = f"Based on the customer request: {request}, provide a technical system design plan and productivity estimation."
    return call_openai(prompt, "system")

def software_engineer(request):
    prompt = f"As a software engineer, estimate the SLOC and effort required for the project: {request}."
    return call_openai(prompt, "software")

def test_engineer(request):
    prompt = f"Write a test strategy for this project: {request}, including test case types and effort estimation."
    return call_openai(prompt, "test")

def documentation_engineer(request):
    prompt = f"As a documentation engineer, create a documentation and effort plan for the educational portal project."
    return call_openai(prompt, "documentation")

In [6]:
class LangGraph:
    def __init__(self):
        self.nodes = []
        self.edges = []
        self.last_message_id = None
        self.message_counter = 0

    def add_message(self, sender, receiver, message):
        self.message_counter += 1
        node_id = self.message_counter
        node = {
            'id': node_id,
            'sender': sender,
            'receiver': receiver,
            'message': message
        }
        self.nodes.append(node)
        if self.last_message_id is not None:
            self.edges.append((self.last_message_id, node_id))
        self.last_message_id = node_id
        return node_id

    def display_graph(self):
        print("Conversation Nodes:")
        for node in self.nodes:
            print(f" [{node['id']}] {node['sender']} -> {node['receiver']}: {node['message']}")
        print("\nMessage Flow (Edges):")
        for edge in self.edges:
            print(f" {edge[0]} -> {edge[1]}")
        print("\n--- End of Conversation Graph ---\n")

In [7]:
import re

def extract_effort_days(text):
    """
    Extracts all numbers followed by 'day(s)' and returns the highest as the total estimated effort.
    """
    matches = re.findall(r"(\d+(?:\.\d+)?)\s+days?", text.lower())
    if matches:
        return max(float(num) for num in matches)
    return 0


def project_manager(customer_request, platform_name, graph):
    print("------LangGraph Experiments ------\n")
    print(f"--- {platform_name}: RUNNING PROJECT PLAN ---\n")

    graph.add_message("Customer", "Project Manager", customer_request)

    total_days = 0  # Accumulator for all effort

    requirements = requirements_engineer(customer_request)
    print(f"Requirements Engineer response:\n{requirements}\n")
    total_days += extract_effort_days(requirements)
    graph.add_message("Project Manager", "Requirements Engineer", customer_request)
    graph.add_message("Requirements Engineer", "Project Manager", requirements)

    design = system_engineer(customer_request)
    print(f"System Engineer response:\n{design}\n")
    total_days += extract_effort_days(design)
    graph.add_message("Project Manager", "System Engineer", customer_request)
    graph.add_message("System Engineer", "Project Manager", design)

    implementation = software_engineer(customer_request)
    print(f"Software Developer response:\n{implementation}\n")
    total_days += extract_effort_days(implementation)
    graph.add_message("Project Manager", "Software Developer", customer_request)
    graph.add_message("Software Developer", "Project Manager", implementation)

    testing = test_engineer(customer_request)
    print(f"Test Engineer response:\n{testing}\n")
    total_days += extract_effort_days(testing)
    graph.add_message("Project Manager", "Test Engineer", customer_request)
    graph.add_message("Test Engineer", "Project Manager", testing)

    docs = documentation_engineer(customer_request)
    print(f"Documentation Engineer response:\n{docs}\n")
    total_days += extract_effort_days(docs)
    graph.add_message("Project Manager", "Documentation Engineer", customer_request)
    graph.add_message("Documentation Engineer", "Project Manager", docs)

    print(f"\n-> Total Estimated Effort to Complete All Phases: {round(total_days)} days\n")
    print("------LangGraph - Run 1 Complete------\n")
    #graph.display_graph()

In [8]:
graph = LangGraph()
customer_request = "Design and build an educational portal platform."
project_manager(customer_request, "LangGraph", graph)

------LangGraph Experiments ------

--- LangGraph: RUNNING PROJECT PLAN ---

Requirements Engineer response:
### Requirements Document for Educational Portal Platform

#### 1. Project Overview
The educational portal platform aims to provide a comprehensive online environment for students, teachers, and administrators to facilitate learning, teaching, and management of educational resources.

---

#### 2. Functional Requirements

**2.1 User Management**
- **FR1**: The system shall allow users to register as students, teachers, or administrators.
- **FR2**: The system shall provide a login/logout functionality for all user types.
- **FR3**: The system shall allow users to reset their passwords.
- **FR4**: The system shall allow administrators to manage user accounts (create, update, delete).

**2.2 Course Management**
- **FR5**: The system shall allow teachers to create and manage courses.
- **FR6**: The system shall allow students to enroll in courses.
- **FR7**: The system shall allow 